# Introduction

**Bridging the Shelter Gap: Measuring Albuquerque’s Unsheltered Disparity (2007–2025)**

## Community Problem
Albuquerque (`NM-500`) faces a growing crisis of unsheltered homelessness—residents living outdoors in vehicles, encampments, and public spaces without reliable nightly shelter. While local agencies and policymakers track annual street counts alongside shelter and housing investments, a critical question remains unanswered for the community: **is Albuquerque’s physical inventory of shelter and supportive housing beds actually keeping pace with street-level need?**

## Data Science Problem Definition
We frame this as a **benchmarking and residual analysis problem** across U.S. Continuums of Care (CoCs) from 2007 to 2025:
* **Target (`y`):** Log-transformed **Unsheltered Homeless** count (`log1p(unsheltered)`), which measures unmet street-level need while preventing mega-cities from skewing the scale.
* **Features (`X`):** Log-transformed year-round bed capacity across three tangible housing tiers—**Emergency Shelter (`ES`)**, **Transitional Housing (`TH`)**, and **Permanent Supportive Housing (`PSH`)**—plus the calendar **year**.
* **Objective:** Train a baseline regression model to estimate the *expected* unsheltered population given a region's bed supply, then evaluate **Albuquerque’s residuals** (`Actual - Predicted`) and compare its per-capita bed coverage against similarly sized peer CoCs to quantify whether physical bed supply explains its unsheltered disparity.

## Project Overview
In this notebook, we combine two official U.S. Department of Housing and Urban Development (HUD) datasets—annual **Point-in-Time (PIT)** street/shelter headcounts and **Housing Inventory Count (HIC)** bed totals. By merging headcounts with tangible bed supply across nearly two decades, we test whether Albuquerque’s unsheltered population reflects a straightforward shortage of beds or a deeper gap between physical housing resources and community need.

# Helpful Links

[PIT and HIC Guidance page
](https://www.hudexchange.info/programs/hdx/pit-hic/#2026-pit-count-and-hic-guidance)

[Homelessness Data Exchange (HDX) Training](https://www.hudexchange.info/trainings/courses/hud-homelessness-data-exchange-hdx-training1/)

[The Annual Homelessness Assessment Reports (AHAR)](https://www.huduser.gov/portal/datasets/ahar.html#:~:text=The%20report%20is%20based%20on%20Homeless%20Management,shelter%20and%20housing%20available%20in%20a%20community.)

# 1. Load the Data


In [ ]:
# Import libraries and modules
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import io
import pandas as pd
import requests

**Data Sources: Excel Workbooks**

[Point-in-Time (PIT) counts](https://www.huduser.gov/portal/datasets/ahar/2025-ahar-part-1-pit-estimates-of-homelessness-in-the-us.html#:~:text=2025%20AHAR%20Report%3A%20Part%201%20%2D%20PIT,ESG%20HDX.%20Research%2C%20Evaluation%2C%20and%20Best%20Practices.)
- Continuum of Care (CoC) is a unit of geography
  - Example: NM-500 is the Albuquerque CoC, while NM-501 covers the Balance of State (the rest of New Mexico outside Albuquerque).
- Point in Time (PIT) count is the annual headcount taken inside that place
  - Example: The annual census/survey of sheltered and unsheltered people experiencing homelessness on a single night in January, conducted within each CoC.

[Housing Inventory Count (HIC)](https://www.huduser.gov/portal/sites/default/files/xls/2007-2025-HIC-Counts-by-CoC.xlsx)
- Housing Inventory Count (HIC) is the annual count of shelter and housing beds inside a CoC
  - Example: The annual snapshot of year-round beds across homeless-dedicated programs in a CoC (like NM-500) on the same night in January as the PIT count.   
  - Bed types (ES, TH, PSH) are the program categories that supply those bedsExample: Emergency Shelter (ES) provides short-term crisis beds, Transitional Housing (TH) offers temporary housing (typically up to 24 months) with support services, and Permanent Supportive Housing (PSH) pairs long-term housing with ongoing wrap-around support.  


PIT - https://www.huduser.gov/portal/sites/default/files/xls/2007-2025-PIT-Counts-by-CoC.xlsb

HIC - https://www.huduser.gov/portal/sites/default/files/xls/2007-2025-HIC-Counts-by-CoC.xlsx

In [ ]:
# Install pyxlsb to read the .xlsb file format
!pip install -q pyxlsb

In [ ]:
# URLs for the data sources
pit_url = "https://www.huduser.gov/portal/sites/default/files/xls/2007-2025-PIT-Counts-by-CoC.xlsb"
hic_url = "https://www.huduser.gov/portal/sites/default/files/xls/2007-2025-HIC-Counts-by-CoC.xlsx"

In [ ]:
# Add a browser User-Agent header so huduser.gov allows the download
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        " (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}

In [ ]:
# Fetch both files
pit_response = requests.get(pit_url, headers=headers)
pit_response.raise_for_status()  # Raises an error if the download fails

hic_response = requests.get(hic_url, headers=headers)
hic_response.raise_for_status()

In [ ]:
# Check if 'PIT' successful
print(f"""
{'='*40}
Checking PIT file
{'='*40}
Status: {pit_response.status_code}
Content-Type: {pit_response.headers.get("Content-Type")}
Content size: {len(pit_response.content)}
""")


Checking PIT file
Status: 200
Content-Type: application/vnd.ms-excel.sheet.binary.macroEnabled.12
Content size: 8859426



In [ ]:
# Check if 'HIC' successful
print(f"""
{'='*40}
Checking HIC file
{'='*40}
Status: {hic_response.status_code}
Content-Type: {hic_response.headers.get("Content-Type")}
Content size: {len(hic_response.content)}
""")


Checking HIC file
Status: 200
Content-Type: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
Content size: 2598803



In [ ]:
# Load the Excel workbooks from memory
pit_excel = pd.ExcelFile(io.BytesIO(pit_response.content), engine="pyxlsb")
hic_excel = pd.ExcelFile(io.BytesIO(hic_response.content))

In [ ]:
# Example: Loop through sheets that represent years (skipping 'Notes' or 'Summary' tabs)
pit_dfs = []
for sheet in pit_excel.sheet_names:
  if sheet.isdigit():  # Checks if the tab name is a year like '2025'
    df = pit_excel.parse(sheet)
    df["Year"] = int(sheet)
    pit_dfs.append(df)

# Stack all years into one master DataFrame
pit_all_years = pd.concat(pit_dfs, ignore_index=True)

# Filter specifically for Albuquerque (NM-500)
abq_pit = pit_all_years[pit_all_years["CoC Number"] == "NM-500"]
display(abq_pit)